# Divar Real Estate Price Prediction Pipeline
**Course/Bootcamp:** QBC13 AI Bootcamp | Project 1  
**Target:** Housing Price Estimation (Sale Total Price & Rent Equivalent Full Credit)  
**Tasks Covered:**
1. Data loading, duplicate cleaning, and initial category inspection
2. Target definition and sale/rent labeling
3. Data cleaning, outlier handling, and year-built validation
4. Feature engineering and leakage prevention
5. Train/test split and preprocessing pipeline
6. Baseline, Random Forest, and LightGBM modeling
7. Test-set evaluation, visualizations, and feature importance


In [ ]:
# ==============================================================================
# Task 1: Data Loading and Deduplication
# ==============================================================================

import pandas as pd
import numpy as np

import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore")

# 1. Find the project root by checking the current directory and its parents
file_name = "preprocessing_output_v4.parquet"
data_path = None

current_dir = Path.cwd().resolve()

for folder in [current_dir, *current_dir.parents]:
    candidate = folder / "data" / "processed" / file_name

    if candidate.exists():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError(
        f"Could not find '{file_name}'.\n"
        f"Current working directory: {current_dir}\n"
        "Expected project structure: data/processed/preprocessing_output_v4.parquet"
    )

# 2. Load the dataset
print(f"Loading data from: {data_path}")
df = pd.read_parquet(data_path)

print(f"Initial dataset shape: {df.shape}")

# 3. Remove an unwanted index column, if present
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)
    print("Dropped unwanted index column: 'Unnamed: 0'")

# 4. Remove exact duplicate rows
duplicate_count = df.duplicated().sum()
df.drop_duplicates(inplace=True)

print(f"Removed duplicate rows: {duplicate_count:,}")
print(f"Dataset shape after deduplication: {df.shape}")

Loading data from: C:\Users\98936\Desktop\divar-housing-analysis\data\processed\preprocessing_output_v4.parquet
Initial dataset shape: (1000000, 98)
Dropped unwanted index column: 'Unnamed: 0'
Removed duplicate rows: 74
Dataset shape after deduplication: (999926, 97)


In [10]:
# ==============================================================================
# Task 2: Target Variable Definition and Sale/Rent Market Separation
# ==============================================================================

# 1. Filter dataset to include only valid standard Sale and Rent records
# In this preprocessed parquet, deal types are represented by binary flags
valid_deal_mask = (df["deal_type_sell"] == 1) | (df["deal_type_rent"] == 1)
df = df[valid_deal_mask].copy()

# 2. Create binary market indicator (1 for Sale, 0 for Rent)
df["is_sale"] = np.where(df["deal_type_sell"] == 1, 1, 0)

# 3. Construct unified target price based on transaction type:
# - Sale: price_value
# - Rent: equivalent_full_credit (full mortgage equivalent)
df["target_price"] = np.where(
    df["is_sale"] == 1, df["price_value"], df["equivalent_full_credit"]
)

# 4. Remove records with missing target price
before_target_clean = len(df)
df = df[df["target_price"].notna()].copy()
dropped_missing_target = before_target_clean - len(df)

print(f"Dataset shape after filtering sale/rent: {df.shape}")
print(f"Dropped listings with missing target price: {dropped_missing_target:,}")
print("\nDistribution across markets:")
print(f" - Sale listings: {(df['is_sale'] == 1).sum():,}")
print(f" - Rent listings: {(df['is_sale'] == 0).sum():,}")

Dataset shape after filtering sale/rent: (901571, 99)
Dropped listings with missing target price: 49,059

Distribution across markets:
 - Sale listings: 553,890
 - Rent listings: 347,681


In [12]:
# ==============================================================================
# Task 3: Data Cleaning, Outlier Handling, and Construction-Year Validation
# ==============================================================================

# 1. Filter strictly positive target prices
df = df[df["target_price"] > 0].copy()

# 2. Filter reasonable building size (10 to 1,500 square meters)
df = df[df["building_size"].between(10, 1500)].copy()

# 3. Handle outliers separately for Sale and Rent markets using 0.5% and 99.5% percentiles
# Separate thresholds prevent sale scale from distorting rent scale
sale_mask = df["is_sale"] == 1
rent_mask = df["is_sale"] == 0

sale_lower = df.loc[sale_mask, "target_price"].quantile(0.005)
sale_upper = df.loc[sale_mask, "target_price"].quantile(0.995)

rent_lower = df.loc[rent_mask, "target_price"].quantile(0.005)
rent_upper = df.loc[rent_mask, "target_price"].quantile(0.995)

print("Target Price Thresholds (0.5% - 99.5%):")
print(f" - Sale Market: {sale_lower:,.0f} to {sale_upper:,.0f} Tomans")
print(f" - Rent Market: {rent_lower:,.0f} to {rent_upper:,.0f} Tomans")

valid_price_mask = (sale_mask & df["target_price"].between(sale_lower, sale_upper)) | (
    rent_mask & df["target_price"].between(rent_lower, rent_upper)
)
df = df[valid_price_mask].copy()

# 4. Validate construction_year (keep realistic solar years 1340 - 1405, preserve NaN for imputer)
valid_year_mask = df["construction_year"].isna() | df["construction_year"].between(
    1340, 1405
)
df = df[valid_year_mask].copy()

print(f"\nDataset shape after cleaning & outlier removal: {df.shape}")
print(f" - Remaining Sale listings: {(df['is_sale'] == 1).sum():,}")
print(f" - Remaining Rent listings: {(df['is_sale'] == 0).sum():,}")

Target Price Thresholds (0.5% - 99.5%):
 - Sale Market: 111,111,111 to 73,536,450,000 Tomans
 - Rent Market: 9,999,998 to 10,000,000,000 Tomans

Dataset shape after cleaning & outlier removal: (863109, 99)
 - Remaining Sale listings: 527,057
 - Remaining Rent listings: 336,052


In [14]:
# ==============================================================================
# Task 4: Feature Engineering and Leakage Prevention
# ==============================================================================

# 1. Target transformation: Log-transform target price to stabilize variance
df["log_target_price"] = np.log1p(df["target_price"])

# 2. Floor ratio feature: position within building height
if "floor" in df.columns and "total_floors_count" in df.columns:
    # Replace negative or zero total_floors with NaN to prevent invalid division
    valid_total_floors = df["total_floors_count"].replace(0, np.nan)
    df["floor_ratio"] = (df["floor"] / valid_total_floors).clip(0, 1.5)
    print("Created feature: 'floor_ratio'")

# 3. Amenity aggregation features
# Resolve exact column names: warehouse, parking, elevator, balcony
parking_col = "PreP_has_parking" if "PreP_has_parking" in df.columns else ("has_parking" if "has_parking" in df.columns else None)
warehouse_col = "PreP_has_warehouse" if "PreP_has_warehouse" in df.columns else ("has_warehouse" if "has_warehouse" in df.columns else None)
elevator_col = "PreP_has_elevator" if "PreP_has_elevator" in df.columns else ("has_elevator" if "has_elevator" in df.columns else None)
balcony_col = "PreP_has_balcony" if "PreP_has_balcony" in df.columns else ("has_balcony" if "has_balcony" in df.columns else None)

basic_cols = [c for c in [parking_col, warehouse_col, elevator_col, balcony_col] if c is not None]
df["basic_amenities"] = df[basic_cols].fillna(0).astype(int).sum(axis=1)

# Luxury amenities score (sum of available luxury flags)
luxury_candidates = [
    "PreP_has_pool", "has_pool",
    "PreP_has_sauna", "has_sauna",
    "PreP_has_jacuzzi", "has_jacuzzi",
    "has_barbecue"
]
# Select unique available luxury columns
selected_luxury_cols = []
for item in ["pool", "sauna", "jacuzzi", "barbecue"]:
    prep_c = f"PreP_has_{item}"
    raw_c = f"has_{item}"
    if prep_c in df.columns:
        selected_luxury_cols.append(prep_c)
    elif raw_c in df.columns:
        selected_luxury_cols.append(raw_c)

df["luxury_score"] = df[selected_luxury_cols].fillna(0).astype(int).sum(axis=1)

print(f"Created 'basic_amenities' using: {basic_cols}")
print(f"Created 'luxury_score' using: {selected_luxury_cols}")

# 4. Leakage Prevention: Drop any financial, rent/credit, or direct price derivatives
leakage_keywords = [
    "price", "rent", "credit", "deposit", "mortgage", "toman", "cost"
]

# Keep our defined targets, drop everything else matching leakage keywords
targets_to_preserve = {"target_price", "log_target_price"}
leakage_cols = [
    col for col in df.columns
    if any(kw in col.lower() for kw in leakage_keywords) and col not in targets_to_preserve
]

print(f"\nIdentified {len(leakage_cols)} potential leakage columns to drop:")
for c in leakage_cols:
    print(f" - {c}")

df.drop(columns=leakage_cols, inplace=True)

print(f"\nDataset shape after feature engineering & leakage removal: {df.shape}")


Created feature: 'floor_ratio'
Created 'basic_amenities' using: ['PreP_has_parking', 'PreP_has_warehouse', 'PreP_has_elevator', 'PreP_has_balcony']
Created 'luxury_score' using: ['PreP_has_pool', 'PreP_has_sauna', 'PreP_has_jacuzzi', 'PreP_has_barbecue']

Identified 17 potential leakage columns to drop:
 - rent_mode
 - rent_value
 - price_mode
 - price_value
 - credit_mode
 - credit_value
 - is_full_mortgage
 - equivalent_full_credit
 - weekend_price_ratio
 - special_day_price_ratio
 - is_rent_credit_convertible
 - log_price_value
 - log_credit_value
 - log_rent_value
 - log_equivalent_full_credit
 - deal_type_rent
 - deal_type_temporary_rent

Dataset shape after feature engineering & leakage removal: (863109, 86)


In [ ]:
# ==============================================================================
# Task 5: Train/Test Split and Preprocessing Pipeline
# ==============================================================================
# 1. Separate Features (X) and Targets (y)
target_cols = ["target_price", "log_target_price"]

# Drop IDs, non-predictive texts, or metadata columns if they exist in df
id_and_text_cols = [
    col
    for col in [
        "token",
        "title",
        "description",
        "post_token",
        "chat_enabled",
        "contact_type",
        "index",
        "level_0",
    ]
    if col in df.columns
]

cols_to_drop_from_X = target_cols + id_and_text_cols
X = df.drop(columns=cols_to_drop_from_X).copy()

y_raw = df["target_price"].copy()
y_log = df["log_target_price"].copy()

print(f"X shape before split: {X.shape}")
print(f"Confirmed 'is_sale' exists in X: {'is_sale' in X.columns}")

# 2. Stratified Train / Test Split (80% Train, 20% Test) based on is_sale
X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y_raw, test_size=0.20, random_state=42, stratify=df["is_sale"]
)

print(f"\nTrain set size: {X_train.shape[0]:,} samples ({X_train.shape[1]} features)")
print(f"Test set size:  {X_test.shape[0]:,} samples ({X_test.shape[1]} features)")
print(f"Train sale ratio: {X_train['is_sale'].mean():.4f}")
print(f"Test sale ratio:  {X_test['is_sale'].mean():.4f}")

# 3. Detect column types for pipeline
categorical_cols = [
    c for c in X.columns if X[c].dtype == "object" or X[c].dtype.name == "category"
]
numerical_cols = [c for c in X.columns if c not in categorical_cols]

print(f"\nNumerical features count:   {len(numerical_cols)}")
print(f"Categorical features count: {len(categorical_cols)}")
if categorical_cols:
    print(f"Categorical features: {categorical_cols}")

# 4. Construct Scikit-Learn Preprocessing Pipeline
# - Numerical: Median imputation + RobustScaler (resistant to real estate outliers)
# - Categorical: Most frequent imputation + OneHotEncoder
numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler())]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="passthrough",
)

print("\n Preprocessing pipeline defined successfully.")

X shape before split: (863109, 82)
Confirmed 'is_sale' exists in X: True

Train set size: 690,487 samples (82 features)
Test set size:  172,622 samples (82 features)
Train sale ratio: 0.6106
Test sale ratio:  0.6107

Numerical features count:   68
Categorical features count: 14
Categorical features: ['has_water', 'has_electricity', 'has_gas', 'PreP_has_pool', 'PreP_has_jacuzzi', 'PreP_has_sauna', 'PreP_has_balcony', 'PreP_has_elevator', 'PreP_has_parking', 'PreP_has_security_guard', 'PreP_has_barbecue', 'PreP_has_warehouse', 'has_utilities', 'has_suana_jacuzzi']

 Preprocessing pipeline defined successfully.


In [ ]:
# ==============================================================================
# Pipeline Sanity Check & Robust Preprocessing
# ==============================================================================

# 1. Drop redundant deal_type_sell if still present
if "deal_type_sell" in X_train.columns:
    X_train = X_train.drop(columns=["deal_type_sell"])
    X_test = X_test.drop(columns=["deal_type_sell"])
    print(" Dropped redundant 'deal_type_sell'.")

print(f"Final features count in X_train: {X_train.shape[1]}")

# 2. Build Robust Preprocessor using make_column_selector
# This guarantees that strings like 'damavand' never enter the numeric imputer!
numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler())]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, make_column_selector(dtype_include=np.number)),
        ("cat", categorical_transformer, make_column_selector(dtype_exclude=np.number)),
    ],
    remainder="drop",
)

# 3. Sanity check: Fit preprocessor on a small subset (5,000 samples)
sample_subset = X_train.sample(5000, random_state=42)
test_transformed = preprocessor.fit_transform(sample_subset)

print(" Sanity test passed successfully!")
print(f"Sample transformed shape (5000 rows): {test_transformed.shape}")

Final features count in X_train: 81
 Sanity test passed successfully!
Sample transformed shape (5000 rows): (5000, 484)
